In [121]:
import pandas as pd 
import numpy as np
import pyreadstat
import os
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test

In [122]:
from scipy.stats import chi2_contingency, mannwhitneyu 
from scipy.stats import chi2 as chi2_dist
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import confusion_matrix

In [123]:
# ── Dossiers de sortie ───────────────────────────────────
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/tables",  exist_ok=True)
plt.rcParams['figure.dpi']        = 130
plt.rcParams['font.family']       = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='Set2')

NAVY  = "#0D1B4B"
BLUE  = "#2196F3"
TEAL  = "#0097A7"
GREEN = "#388E3C"
CORAL = "#FF5722"
AMBER = "#F57C00"
GRAY  = "#90A4AE"
print("✓ Bibliothèques chargées !")
print("✓ Dossiers outputs/figures/ et outputs/tables/ prêts")

✓ Bibliothèques chargées !
✓ Dossiers outputs/figures/ et outputs/tables/ prêts


In [124]:
# ── 1.1 Chargement du fichier SAV ───────────────────────
df_raw, meta = pyreadstat.read_sav("../data/CMIR71FL.SAV")
df_raw.columns = df_raw.columns.str.lower()
print(f"Données brutes : {df_raw.shape[0]:,} femmes · {df_raw.shape[1]} variables")
# ── 1.2 Sélection et renommage des variables ────────────
variables_dhs = {
    'v005': 'poids_sondage',
    'v012': 'age',
    'v106': 'niveau_instruction',
    'v201': 'nb_enfants_total',
    'v218': 'nb_enfants_vivants',
    'v313': 'utilisation_contraceptif',
    'v501': 'statut_matrimonial',
    'v025': 'milieu_residence',
    'v024': 'region',
    'v130': 'religion',
    'v714': 'travail',
    'v511': 'age_premier_mariage',
    'v190': 'quintile_richesse',
    'v602': 'desir_enfant',
}

# On ne garde que les colonnes qui existent dans le fichier
cols_ok = [c for c in variables_dhs if c in df_raw.columns]
df = df_raw[cols_ok].rename(columns=variables_dhs).copy()

print(f"Variables sélectionnées : {len(df.columns)}")
print(f"Colonnes : {list(df.columns)}")

Données brutes : 14,677 femmes · 5102 variables
Variables sélectionnées : 14
Colonnes : ['poids_sondage', 'age', 'niveau_instruction', 'nb_enfants_total', 'nb_enfants_vivants', 'utilisation_contraceptif', 'statut_matrimonial', 'milieu_residence', 'region', 'religion', 'travail', 'age_premier_mariage', 'quintile_richesse', 'desir_enfant']


In [125]:
# ── 1.3 Binarisation de v602 ────────────────────────────
df['desir_bin'] = df['desir_enfant'].apply(
    lambda x: 1 if x in [1, 2, 3] else (0 if x in [4, 5, 6, 7, 8] else np.nan)
)

# Supprimer les lignes sans réponse valide
df = df[df['desir_bin'].notna()].copy()
df['desir_bin'] = df['desir_bin'].astype(int)

# Statistiques de base
N     = len(df)
N_OUI = int(df['desir_bin'].sum())
N_NON = N - N_OUI

print(f"Effectif final   : {N:,} femmes")
print(f"Désire un enfant : {N_OUI:,} ({N_OUI/N*100:.1f}%)")
print(f"Ne désire pas    : {N_NON:,} ({N_NON/N*100:.1f}%)")
print(f"\n⚠ Déséquilibre : {N_OUI/N*100:.1f}% vs {N_NON/N*100:.1f}%")
print("   → Ce déséquilibre sera traité avec SMOTE dans le notebook ML")

Effectif final   : 13,527 femmes
Désire un enfant : 12,782 (94.5%)
Ne désire pas    : 745 (5.5%)

⚠ Déséquilibre : 94.5% vs 5.5%
   → Ce déséquilibre sera traité avec SMOTE dans le notebook ML


In [126]:
# ── 1.4 Variables dérivées ──────────────────────────────

# Poids de sondage normalisé
df['poids'] = df['poids_sondage'] / 1_000_000

# Contraceptif binaire (toute méthode = 1, aucune = 0)
df['contraceptif_bin'] = (df['utilisation_contraceptif'] > 0).astype(int)

# Région septentrionale
# ✅ Codes DHS Cameroun 2018 : Adamaoua=1 · Extrême-Nord=4 · Nord=6
df['region_nord'] = df['region'].isin([1, 4, 6]).astype(int)

# Religion musulmane
# ✅ Code DHS Cameroun 2018 : Musulman=4
df['religion_musulman'] = (df['religion'] == 4).astype(int)

print("Variables dérivées créées :")
print(f"  contraceptif_bin  : {df['contraceptif_bin'].sum():,} utilisatrices ({df['contraceptif_bin'].mean()*100:.1f}%)")
print(f"  region_nord       : {df['region_nord'].sum():,} femmes ({df['region_nord'].mean()*100:.1f}%)")
print(f"  religion_musulman : {df['religion_musulman'].sum():,} femmes ({df['religion_musulman'].mean()*100:.1f}%)")

Variables dérivées créées :
  contraceptif_bin  : 2,740 utilisatrices (20.3%)
  region_nord       : 2,999 femmes (22.2%)
  religion_musulman : 3,059 femmes (22.6%)


In [127]:
# ── 1.5 Tranches d'âge ──────────────────────────────────
df['tranche_age'] = pd.cut(
    df['age'],
    bins=[14, 19, 24, 29, 34, 39, 44, 49],
    labels=['15-19','20-24','25-29','30-34','35-39','40-44','45-49']
)

# ── Labels lisibles ──────────────────────────────────────
df['instruction_lbl']   = df['niveau_instruction'].map(
    {0:'Aucun', 1:'Primaire', 2:'Secondaire', 3:'Supérieur'})
df['residence_lbl']     = df['milieu_residence'].map({1:'Urbain', 2:'Rural'})
df['quintile_lbl']      = df['quintile_richesse'].map(
    {1:'Le plus pauvre', 2:'Pauvre', 3:'Moyen', 4:'Riche', 5:'Le plus riche'})
df['contraceptif_lbl']  = df['contraceptif_bin'].map({0:"N'utilise pas", 1:'Utilise'})
df['statut_mat_lbl']    = df['statut_matrimonial'].map(
    {0:'Jamais marié(e)', 1:'Marié(e)', 2:'En union libre',
     3:'Veuf/Veuve', 4:'Divorcé(e)', 5:'Séparé(e)'})
df['religion_lbl']      = df['religion'].map(
    {1:'Catholique', 2:'Protestant', 3:'Autres chrétiens',
     4:'Musulman', 5:'Animiste', 6:'Sans religion', 96:'Autre'})
df['travail_lbl']       = df['travail'].map({0:'Non', 1:'Oui'})
df['region_nord_lbl']   = df['region_nord'].map({1:'Septentrionale', 0:'Autre région'})
df['religion_musl_lbl'] = df['religion_musulman'].map({1:'Musulmane', 0:'Non musulmane'})
df['desir_lbl']         = df['desir_bin'].map({1:'Désire un enfant', 0:'Ne désire pas'})

print("✓ Labels créés pour tous les graphiques")
print(f"\nAperçu du DataFrame final :")
df[['age','desir_bin','instruction_lbl','residence_lbl',
    'region_nord','religion_musulman']].head()

✓ Labels créés pour tous les graphiques

Aperçu du DataFrame final :


,age,desir_bin,instruction_lbl,residence_lbl,region_nord,religion_musulman
0,31.0,1,Secondaire,Urbain,0,0
1,19.0,1,Aucun,Urbain,0,0
2,22.0,1,Secondaire,Urbain,0,1
3,16.0,1,Primaire,Urbain,0,1
4,17.0,1,Aucun,Urbain,0,1


In [128]:
# ── 2.1 Variables continues ─────────────────────────────
print("=" * 55)
print("TABLE 1A — STATISTIQUES DES VARIABLES CONTINUES")
print("=" * 55)

desc = df[['age','nb_enfants_vivants','nb_enfants_total']].describe().round(2)
desc.index = ['N','Moyenne','Écart-type','Min','Q1','Médiane','Q3','Max']
desc.columns = ['Âge (années)','Nb enfants vivants','Nb enfants total']
print(desc.to_string())

desc.to_csv("../outputs/tables/01a_stats_continues.csv")
print("\n✓ Sauvegardé → 01a_stats_continues.csv")

TABLE 1A — STATISTIQUES DES VARIABLES CONTINUES
            Âge (années)  Nb enfants vivants  Nb enfants total
N               13527.00            13527.00          13527.00
Moyenne            27.80                2.25              2.51
Écart-type          9.44                2.30              2.62
Min                15.00                0.00              0.00
Q1                 20.00                0.00              0.00
Médiane            26.00                2.00              2.00
Q3                 35.00                4.00              4.00
Max                49.00               13.00             16.00

✓ Sauvegardé → 01a_stats_continues.csv


In [129]:
# ── 2.2 Fonction fréquences ─────────────────────────────
def freq_table(serie, nom_colonne):
    """Calcule effectif et % pour une variable catégorielle.
    
    Paramètres :
        serie        : la colonne pandas à analyser
        nom_colonne  : le nom à donner à la colonne des modalités
    Retourne :
        DataFrame avec Modalité / Effectif / Pourcentage (%)
    """
    counts = serie.value_counts(dropna=True)
    pcts   = serie.value_counts(normalize=True, dropna=True) * 100
    t = pd.DataFrame({
        nom_colonne       : counts.index.astype(str),
        'Effectif'        : counts.values,
        'Pourcentage (%)' : pcts.values.round(1)
    })
    return t.reset_index(drop=True)

# Test rapide
print(freq_table(df['residence_lbl'], 'Milieu').to_string(index=False))

Milieu  Effectif  Pourcentage (%)
Urbain      7343             54.3
 Rural      6184             45.7


In [130]:
# ── 2.3 Table 1b complète ───────────────────────────────
print("=" * 55)
print("TABLE 1B — VARIABLES CATÉGORIELLES")
print("=" * 55)

sections = {}

# Tranche d'âge
sections["Tranche d'âge"] = freq_table(df['tranche_age'], "Tranche d'âge")

# Instruction (ordre logique : Aucun → Supérieur)
ordre_i = ['Aucun','Primaire','Secondaire','Supérieur']
t = freq_table(df['instruction_lbl'], "Instruction")
t['_ord'] = t['Instruction'].map({v:i for i,v in enumerate(ordre_i)})
sections["Niveau d'instruction"] = t.sort_values('_ord').drop('_ord',axis=1).reset_index(drop=True)

# Résidence
sections["Milieu de résidence"] = freq_table(df['residence_lbl'], "Résidence")

# Statut matrimonial
sections["Statut matrimonial"] = freq_table(df['statut_mat_lbl'], "Statut")

# Religion
sections["Religion"] = freq_table(df['religion_lbl'], "Religion")

# Quintile (ordre logique)
ordre_q = ['Le plus pauvre','Pauvre','Moyen','Riche','Le plus riche']
t = freq_table(df['quintile_lbl'], "Quintile")
t['_ord'] = t['Quintile'].map({v:i for i,v in enumerate(ordre_q)})
sections["Quintile de richesse"] = t.sort_values('_ord').drop('_ord',axis=1).reset_index(drop=True)

# Contraceptif
sections["Utilisation contraceptif"] = freq_table(df['contraceptif_lbl'], "Contraceptif")

# Travail
sections["Emploi (travail)"] = freq_table(df['travail_lbl'], "Travail")

# Affichage de toutes les sections
for nom, t in sections.items():
    print(f"\n--- {nom} ---")
    print(t.to_string(index=False))

# Sauvegarde
all_rows = []
for nom, t in sections.items():
    for _, row in t.iterrows():
        all_rows.append({'Variable': nom, **row.to_dict()})
pd.DataFrame(all_rows).to_csv("../outputs/tables/01b_stats_categories.csv", index=False)
print("\n✓ Sauvegardé → 01b_stats_categories.csv")

TABLE 1B — VARIABLES CATÉGORIELLES

--- Tranche d'âge ---
Tranche d'âge  Effectif  Pourcentage (%)
        15-19      3349             24.8
        20-24      2463             18.2
        25-29      2345             17.3
        30-34      1884             13.9
        35-39      1485             11.0
        40-44      1091              8.1
        45-49       910              6.7

--- Niveau d'instruction ---
Instruction  Effectif  Pourcentage (%)
      Aucun      2364             17.5
   Primaire      3787             28.0
 Secondaire      6400             47.3
  Supérieur       976              7.2

--- Milieu de résidence ---
Résidence  Effectif  Pourcentage (%)
   Urbain      7343             54.3
    Rural      6184             45.7

--- Statut matrimonial ---
         Statut  Effectif  Pourcentage (%)
       Marié(e)      5514             40.8
Jamais marié(e)      4856             35.9
 En union libre      1949             14.4
      Séparé(e)       646              4.8
     V

In [131]:
# ── Figure 1 : Distribution de la variable cible ────────
fig, ax = plt.subplots(figsize=(7, 5))
counts = df['desir_lbl'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=[BLUE, CORAL], edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    pct = val / N * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11)
ax.set_title("Distribution du désir d'avoir un autre enfant\n(Cameroun EDSC 2018)",
             fontsize=13, fontweight='bold')
ax.set_ylabel("Nombre de femmes")
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.savefig("../outputs/figures/fig1_variable_cible.png", bbox_inches='tight')
plt.show()
print("✓ Figure 1 sauvegardée")

✓ Figure 1 sauvegardée


In [132]:
# ── Figure 2 : Désir par tranche d'âge ──────────────────
# On calcule le % désirant un enfant dans chaque tranche d'âge
fig, ax = plt.subplots(figsize=(10, 5))
age_d = df.groupby('tranche_age', observed=True)['desir_bin'].mean() * 100
bars  = ax.bar(age_d.index.astype(str), age_d.values,
               color=BLUE, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, age_d.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_title("Désir d'un autre enfant par tranche d'âge (%)", fontsize=13, fontweight='bold')
ax.set_ylabel("% désirant un autre enfant")
ax.set_xlabel("Tranche d'âge")
ax.set_ylim(0, 108)
plt.tight_layout()
plt.savefig("../outputs/figures/fig2_desir_par_age.png", bbox_inches='tight')
plt.show()
print("✓ Figure 2 sauvegardée")

✓ Figure 2 sauvegardée


In [133]:
# ── Figure 3 : Désir par niveau d'instruction ────────────
fig, ax = plt.subplots(figsize=(8, 5))
ordre = ['Aucun','Primaire','Secondaire','Supérieur']
instr_d = df.groupby('instruction_lbl', observed=True)['desir_bin'].mean() * 100
instr_d = instr_d.reindex([x for x in ordre if x in instr_d.index])
bars = ax.bar(instr_d.index, instr_d.values,
              color=GREEN, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, instr_d.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11)
ax.set_title("Désir d'un autre enfant par niveau d'instruction (%)",
             fontsize=13, fontweight='bold')
ax.set_ylabel("% désirant un autre enfant")
ax.set_ylim(0, 108)
plt.tight_layout()
plt.savefig("../outputs/figures/fig3_desir_par_instruction.png", bbox_inches='tight')
plt.show()
print("✓ Figure 3 sauvegardée")

✓ Figure 3 sauvegardée


In [134]:
# ── Figure 4 : Désir par nb d'enfants vivants ────────────
fig, ax = plt.subplots(figsize=(10, 5))
nbe = df[df['nb_enfants_vivants'] <= 8].groupby(
    'nb_enfants_vivants')['desir_bin'].mean() * 100
bars = ax.bar(nbe.index.astype(str), nbe.values,
              color=AMBER, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, nbe.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_title("Désir d'un autre enfant par nombre d'enfants vivants (%)",
             fontsize=13, fontweight='bold')
ax.set_ylabel("% désirant un autre enfant")
ax.set_xlabel("Nombre d'enfants vivants")
ax.set_ylim(0, 108)
plt.tight_layout()
plt.savefig("../outputs/figures/fig4_desir_par_nb_enfants.png", bbox_inches='tight')
plt.show()
print("✓ Figure 4 sauvegardée")

✓ Figure 4 sauvegardée


In [135]:
# ── Figure 5 : Résidence + Contraceptif ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gauche : par résidence
res_d = df.groupby('residence_lbl', observed=True)['desir_bin'].mean() * 100
b1 = axes[0].bar(res_d.index, res_d.values,
                 color=['#7B1FA2','#C2185B'], edgecolor='white', linewidth=1.5)
for bar, val in zip(b1, res_d.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=11)
axes[0].set_title("Par milieu de résidence", fontsize=12, fontweight='bold')
axes[0].set_ylabel("% désirant un autre enfant")
axes[0].set_ylim(0, 105)

# Droite : par contraceptif
con_d = df.groupby('contraceptif_lbl', observed=True)['desir_bin'].mean() * 100
b2 = axes[1].bar(con_d.index, con_d.values,
                 color=[TEAL, CORAL], edgecolor='white', linewidth=1.5)
for bar, val in zip(b2, con_d.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=11)
axes[1].set_title("Par utilisation de contraceptifs", fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 105)

fig.suptitle("Désir d'un autre enfant selon le contexte", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig5_residence_contraceptif.png", bbox_inches='tight')
plt.show()
print("✓ Figure 5 sauvegardée")

✓ Figure 5 sauvegardée


In [136]:
# ── Figure 6 : Heatmap de corrélation ───────────────────
fig, ax = plt.subplots(figsize=(9, 7))
vars_corr = ['desir_bin','age','niveau_instruction','nb_enfants_vivants',
             'contraceptif_bin','milieu_residence','quintile_richesse']
corr_df = df[vars_corr].dropna().corr().round(2)
labels  = ['Désir enfant','Âge','Instruction','Nb enfants\nvivants',
           'Contraceptif','Milieu\nrésidence','Quintile\nrichesse']
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=labels, yticklabels=labels, ax=ax,
            linewidths=0.5, square=True)
ax.set_title("Matrice de corrélation entre les variables", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig6_heatmap_correlation.png", bbox_inches='tight')
plt.show()

# Afficher les corrélations importantes
print("Corrélations à noter :")
print(f"  désir ↔ âge          : {corr_df.loc['desir_bin','age']:.2f}  (négatif = plus on est âgé, moins on désire)")
print(f"  âge ↔ nb enfants     : {corr_df.loc['age','nb_enfants_vivants']:.2f}  (élevé → multicolinéarité potentielle)")
print(f"  résidence ↔ richesse : {corr_df.loc['milieu_residence','quintile_richesse']:.2f}")
print("✓ Figure 6 sauvegardée")

Corrélations à noter :
  désir ↔ âge          : -0.27  (négatif = plus on est âgé, moins on désire)
  âge ↔ nb enfants     : 0.68  (élevé → multicolinéarité potentielle)
  résidence ↔ richesse : -0.67
✓ Figure 6 sauvegardée


In [137]:
# ── Figure A : Analyses croisées ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauche : désir par âge ET résidence
age_res = df.groupby(['tranche_age','residence_lbl'], observed=True)['desir_bin'].mean() * 100
age_res.unstack('residence_lbl').plot(kind='bar', ax=axes[0],
    color=['#1565C0','#E65100'], edgecolor='white')
axes[0].set_title("Désir d'un enfant par âge et résidence (%)", fontsize=11, fontweight='bold')
axes[0].set_ylabel("% désirant un autre enfant")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title='Milieu')
for p in axes[0].patches:
    if p.get_height() > 0:
        axes[0].annotate(f"{p.get_height():.0f}%",
                         (p.get_x()+p.get_width()/2, p.get_height()),
                         ha='center', va='bottom', fontsize=7)

# Droite : désir par quintile ET instruction
ordre_q = ['Le plus pauvre','Pauvre','Moyen','Riche','Le plus riche']
q_i = df.groupby(['quintile_lbl','instruction_lbl'], observed=True)['desir_bin'].mean() * 100
q_i = q_i.unstack('instruction_lbl').reindex([x for x in ordre_q if x in q_i.index])
q_i.plot(kind='bar', ax=axes[1], edgecolor='white')
axes[1].set_title("Désir d'un enfant par quintile et instruction (%)", fontsize=11, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15, ha='right')
axes[1].legend(title='Instruction', fontsize=8)

fig.suptitle("Analyse croisée du désir d'avoir un autre enfant\n(EDSC Cameroun 2018)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/figA_croise_age_residence_quintile.png", bbox_inches='tight')
plt.show()
print("✓ Figure A sauvegardée")

✓ Figure A sauvegardée


In [138]:
# ── Figure B : Boxplots ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
vars_box = [('age','Âge (années)'),
            ('nb_enfants_vivants','Nb enfants vivants'),
            ('nb_enfants_total','Nb enfants total')]
palette = {'Ne désire pas':'#E57373', 'Désire un enfant':'#42A5F5'}
for ax, (var, label) in zip(axes, vars_box):
    sns.boxplot(data=df, x='desir_lbl', y=var, ax=ax, palette=palette)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=10, ha='right')
fig.suptitle("Distribution des variables continues selon le désir d'un autre enfant\n(EDSC Cameroun 2018)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/figB_boxplots_variables_continues.png", bbox_inches='tight')
plt.show()
print("✓ Figure B sauvegardée")
print("\nLecture rapide :")
print(f"  Âge médian (désire)     : {df[df['desir_bin']==1]['age'].median():.0f} ans")
print(f"  Âge médian (ne désire)  : {df[df['desir_bin']==0]['age'].median():.0f} ans")

✓ Figure B sauvegardée

Lecture rapide :
  Âge médian (désire)     : 26 ans
  Âge médian (ne désire)  : 42 ans


In [139]:
# ── Table 3a : Tests Chi² pour les 10 variables ─────────
vars_chi2 = {
    "Tranche d'âge"         : 'tranche_age',
    "Niveau d'instruction"  : 'instruction_lbl',
    "Milieu de résidence"   : 'residence_lbl',
    "Statut matrimonial"    : 'statut_mat_lbl',
    "Religion"              : 'religion_lbl',
    "Quintile de richesse"  : 'quintile_lbl',
    "Utilisation contraceptif": 'contraceptif_lbl',
    "Emploi (travail)"      : 'travail_lbl',
    "Région septentrionale" : 'region_nord_lbl',
    "Religion musulmane"    : 'religion_musl_lbl',
}

def cramers_v(chi2_val, n, k, r):
    """Calcule le V de Cramér (force de l'association)."""
    return np.sqrt(chi2_val / (n * (min(k, r) - 1)))

print(f"{'Variable':<30} {'Chi²':>10} {'ddl':>5} {'p-value':>10} {'V Cramér':>10} {'Sig.':>5}")
print("-" * 72)

rows_chi2 = []
for label, col in vars_chi2.items():
    if col not in df.columns:
        continue
    tab = pd.crosstab(df[col], df['desir_bin'])
    chi2_val, p, dof, _ = chi2_contingency(tab)
    r, k = tab.shape
    v    = cramers_v(chi2_val, N, k, r)
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'NS'))
    force = ('Très forte' if v > 0.5 else ('Forte' if v > 0.3 else ('Modérée' if v > 0.1 else 'Faible')))
    print(f"{label:<30} {chi2_val:>10.2f} {dof:>5} {p:>10.4f} {v:>10.3f} {sig:>5}")
    rows_chi2.append({'Variable':label,'Chi²':round(chi2_val,2),'ddl':dof,
                      'p-value':round(p,4),"V Cramér":round(v,3),'Sig.':sig,'Force':force})

chi2_df = pd.DataFrame(rows_chi2)
chi2_df.to_csv("../outputs/tables/03a_tests_chi2.csv", index=False)
print("\n✓ Sauvegardé → 03a_tests_chi2.csv")

Variable                             Chi²   ddl    p-value   V Cramér  Sig.
------------------------------------------------------------------------
Tranche d'âge                     1605.85     6     0.0000      0.345   ***
Niveau d'instruction                80.45     3     0.0000      0.077   ***
Milieu de résidence                 21.24     1     0.0000      0.040   ***
Statut matrimonial                 282.96     5     0.0000      0.145   ***
Religion                             4.10     5     0.5355      0.017    NS
Quintile de richesse                31.56     4     0.0000      0.048   ***
Utilisation contraceptif            50.00     1     0.0000      0.061   ***
Emploi (travail)                    69.46     1     0.0000      0.072   ***
Région septentrionale                0.37     1     0.5451      0.005    NS
Religion musulmane                   0.29     1     0.5904      0.005    NS

✓ Sauvegardé → 03a_tests_chi2.csv


In [140]:
# ── Table 3b : Tests Mann-Whitney ───────────────────────
print(f"{'Variable':<25} {'Moy. Désire':>12} {'Moy. Ne désire':>15} {'p-value':>10} {'Sig.':>5}")
print("-" * 68)

rows_mw = []
for var, label in [('age','Âge (années)'),
                   ('nb_enfants_vivants','Nb enfants vivants'),
                   ('nb_enfants_total','Nb enfants total')]:
    g1   = df[df['desir_bin'] == 1][var].dropna()
    g0   = df[df['desir_bin'] == 0][var].dropna()
    stat, p = mannwhitneyu(g1, g0, alternative='two-sided')
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'NS'))
    print(f"{label:<25} {g1.mean():>12.2f} {g0.mean():>15.2f} {p:>10.4f} {sig:>5}")
    rows_mw.append({'Variable':label, 'Moy. (Désire)':round(g1.mean(),2),
                    'Moy. (Ne désire pas)':round(g0.mean(),2),
                    'p-value':round(p,4), 'Sig.':sig})

mw_df = pd.DataFrame(rows_mw)
mw_df.to_csv("../outputs/tables/03b_mann_whitney.csv", index=False)
print("\n✓ Sauvegardé → 03b_mann_whitney.csv")

Variable                   Moy. Désire  Moy. Ne désire    p-value  Sig.
--------------------------------------------------------------------
Âge (années)                     27.19           38.38     0.0000   ***
Nb enfants vivants                2.19            3.16     0.0000   ***
Nb enfants total                  2.44            3.70     0.0000   ***

✓ Sauvegardé → 03b_mann_whitney.csv


In [141]:
# ── 5.1 Définition des 10 variables du modèle ───────────
VARS_MODELE = [
    'age',               # v012
    'niveau_instruction', # v106
    'nb_enfants_vivants', # v218
    'contraceptif_bin',   # v313
    'statut_matrimonial', # v501
    'milieu_residence',   # v025
    'quintile_richesse',  # v190
    'travail',            # v714
    'region_nord',        # v024
    'religion_musulman',  # v130
]

LABELS = {
    'const'              : 'Constante',
    'age'                : 'Âge',
    'niveau_instruction' : "Niveau d'instruction",
    'nb_enfants_vivants' : 'Nb enfants vivants',
    'contraceptif_bin'   : 'Utilisation contraceptif',
    'statut_matrimonial' : 'Statut matrimonial',
    'milieu_residence'   : 'Milieu de résidence',
    'quintile_richesse'  : 'Quintile de richesse',
    'travail'            : 'Emploi (travail)',
    'region_nord'        : 'Région septentrionale',
    'religion_musulman'  : 'Religion musulmane',
}

# Sous-ensemble sans valeurs manquantes
df_mod = df[VARS_MODELE + ['desir_bin']].dropna().copy()
X = sm.add_constant(df_mod[VARS_MODELE].astype(float))
y = df_mod['desir_bin'].astype(float)

print(f"Effectif pour la régression : {len(df_mod):,} observations")
print(f"Variables dans le modèle    : {len(VARS_MODELE)}")

Effectif pour la régression : 13,527 observations
Variables dans le modèle    : 10


In [142]:
# ── 5.2 Estimation du modèle ────────────────────────────
# statsmodels estime les coefficients par maximum de vraisemblance
logit_model = Logit(y, X).fit(disp=False)

# Extraction des résultats
params = logit_model.params    # Coefficients β
conf   = logit_model.conf_int() # Intervalles de confiance
pvals  = logit_model.pvalues    # p-values

# ── Pseudo R²  ───────────────────────────────────────────
n          = len(y)
llnull     = logit_model.llnull   # Log-vraisemblance du modèle nul
llmodel    = logit_model.llf      # Log-vraisemblance du modèle estimé
mcfadden   = logit_model.prsquared
cox_snell  = 1 - np.exp(2 * (llnull - llmodel) / n)
nagelkerke = cox_snell / (1 - np.exp(2 * llnull / n))

print("=== Indicateurs du modèle ===")
print(f"  N observations           : {int(logit_model.nobs):,}")
print(f"  Pseudo R² McFadden       : {mcfadden:.4f}  ({mcfadden*100:.1f}%)")
print(f"  Pseudo R² Cox-Snell      : {cox_snell:.4f}  ({cox_snell*100:.1f}%)")
print(f"  Pseudo R² Nagelkerke     : {nagelkerke:.4f}  ({nagelkerke*100:.1f}%)")
print(f"  AIC                      : {logit_model.aic:.2f}")
print(f"  BIC                      : {logit_model.bic:.2f}")

=== Indicateurs du modèle ===
  N observations           : 13,527
  Pseudo R² McFadden       : 0.1932  (19.3%)
  Pseudo R² Cox-Snell      : 0.0791  (7.9%)
  Pseudo R² Nagelkerke     : 0.2278  (22.8%)
  AIC                      : 4675.26
  BIC                      : 4757.90


In [143]:
# ── 5.3 Table 4 : Odds Ratios ───────────────────────────
print("=" * 78)
print("TABLE 4 — ODDS RATIOS AJUSTÉS (Régression logistique binaire)")
print("=" * 78)
print(f"{'Variable':<28} {'β':>8} {'OR':>8} {'IC inf.':>9} {'IC sup.':>9} {'p-value':>10} {'Sig.':>5}")
print("-" * 78)

or_rows = []
for var in params.index:
    OR     = np.exp(params[var])
    ic_inf = np.exp(conf.loc[var, 0])
    ic_sup = np.exp(conf.loc[var, 1])
    p      = pvals[var]
    sig    = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'NS'))
    label  = LABELS.get(var, var)
    print(f"{label:<28} {params[var]:>8.4f} {OR:>8.4f} {ic_inf:>9.4f} {ic_sup:>9.4f} {p:>10.4f} {sig:>5}")
    or_rows.append({'Variable':label, 'β':round(params[var],4),
                    'OR':round(OR,4), 'IC inf.':round(ic_inf,4),
                    'IC sup.':round(ic_sup,4), 'p-value':round(p,4), 'Sig.':sig})

or_df = pd.DataFrame(or_rows)
or_df.to_csv("../outputs/tables/04_odds_ratio.csv", index=False)
print("\n✓ Sauvegardé → 04_odds_ratio.csv")

TABLE 4 — ODDS RATIOS AJUSTÉS (Régression logistique binaire)
Variable                            β       OR   IC inf.   IC sup.    p-value  Sig.
------------------------------------------------------------------------------
Constante                      7.1766 1308.5118  682.6987 2507.9922     0.0000   ***
Âge                           -0.1475   0.8628    0.8537    0.8720     0.0000   ***
Niveau d'instruction           0.1307   1.1397    1.0060    1.2911     0.0400     *
Nb enfants vivants             0.1994   1.2206    1.1740    1.2691     0.0000   ***
Utilisation contraceptif       0.5665   1.7620    1.3633    2.2773     0.0000   ***
Statut matrimonial            -0.0308   0.9697    0.9119    1.0312     0.3267    NS
Milieu de résidence           -0.3334   0.7165    0.5771    0.8895     0.0025    **
Quintile de richesse           0.0458   1.0469    0.9549    1.1478     0.3289    NS
Emploi (travail)               0.0775   1.0806    0.8920    1.3091     0.4283    NS
Région septentrion

In [144]:
# ── 5.4 Hosmer-Lemeshow ─────────────────────────────────
y_pred_prob = logit_model.predict(X)
n_groups    = 10
quantiles   = pd.qcut(y_pred_prob, n_groups, duplicates='drop')
hl          = df_mod.copy()
hl['pred']  = y_pred_prob.values
hl['grp']   = quantiles.values

hl_t = hl.groupby('grp', observed=True).agg(
    n=('desir_bin','count'), obs1=('desir_bin','sum'),
    mean_pred=('pred','mean')).reset_index()
hl_t['exp1'] = hl_t['mean_pred'] * hl_t['n']
hl_t['obs0'] = hl_t['n'] - hl_t['obs1']
hl_t['exp0'] = hl_t['n'] - hl_t['exp1']

hl_chi2 = (((hl_t['obs1']-hl_t['exp1'])**2/hl_t['exp1']) +
           ((hl_t['obs0']-hl_t['exp0'])**2/hl_t['exp0'])).sum()
hl_p = 1 - chi2_dist.cdf(hl_chi2, df=n_groups-2)

print(f"Test de Hosmer-Lemeshow :")
print(f"  Chi² = {hl_chi2:.4f}  |  p = {hl_p:.4f}")
print(f"  → {'✓ Bon ajustement (p > 0.05)' if hl_p > 0.05 else '⚠ Ajustement à vérifier (p ≤ 0.05)'}")

# ── Tableau de classification ────────────────────────────
y_pred_class = (y_pred_prob >= 0.5).astype(int)
cm = confusion_matrix(y, y_pred_class)
tn, fp, fn, tp = cm.ravel()
sensib  = tp / (tp + fn) * 100
specif  = tn / (tn + fp) * 100
pct_ok  = (tp + tn) / len(y) * 100

print(f"\nTableau de classification (seuil = 0,5) :")
print(f"  Vrais Positifs  (VP) : {tp:,}   → Sensibilité  : {sensib:.1f}%")
print(f"  Vrais Négatifs  (VN) : {tn:,}     → Spécificité  : {specif:.1f}%")
print(f"  Faux Positifs   (FP) : {fp:,}")
print(f"  Faux Négatifs   (FN) : {fn:,}")
print(f"  % bien classés       : {pct_ok:.1f}%")

# Export indicateurs
indic_rows = [
    {'Indicateur':'N observations',         'Valeur':int(logit_model.nobs)},
    {'Indicateur':'Log-vraisemblance',       'Valeur':round(llmodel,4)},
    {'Indicateur':'AIC',                     'Valeur':round(logit_model.aic,2)},
    {'Indicateur':'BIC',                     'Valeur':round(logit_model.bic,2)},
    {'Indicateur':'Pseudo R² McFadden',      'Valeur':round(mcfadden,4)},
    {'Indicateur':'Pseudo R² Cox-Snell',     'Valeur':round(cox_snell,4)},
    {'Indicateur':'Pseudo R² Nagelkerke',    'Valeur':round(nagelkerke,4)},
    {'Indicateur':'Hosmer-Lemeshow Chi²',    'Valeur':round(hl_chi2,4)},
    {'Indicateur':'Hosmer-Lemeshow p-value', 'Valeur':round(hl_p,4)},
    {'Indicateur':'Sensibilité (%)',         'Valeur':round(sensib,1)},
    {'Indicateur':'Spécificité (%)',         'Valeur':round(specif,1)},
    {'Indicateur':'% bien classés',          'Valeur':round(pct_ok,1)},
]
pd.DataFrame(indic_rows).to_csv("../outputs/tables/04b_indicateurs_modele.csv", index=False)
print("\n✓ Sauvegardé → 04b_indicateurs_modele.csv")

Test de Hosmer-Lemeshow :
  Chi² = 73.8608  |  p = 0.0000
  → ⚠ Ajustement à vérifier (p ≤ 0.05)

Tableau de classification (seuil = 0,5) :
  Vrais Positifs  (VP) : 12,770   → Sensibilité  : 99.9%
  Vrais Négatifs  (VN) : 23     → Spécificité  : 3.1%
  Faux Positifs   (FP) : 722
  Faux Négatifs   (FN) : 12
  % bien classés       : 94.6%

✓ Sauvegardé → 04b_indicateurs_modele.csv


In [145]:
# ── Figure C : Forest Plot ───────────────────────────────
# Le forest plot visualise les OR avec leurs intervalles de confiance
# Point GAUCHE de la ligne OR=1 → réduit le désir
# Point DROITE de la ligne OR=1 → augmente le désir
fig, ax = plt.subplots(figsize=(11, 7))
or_plot = or_df[or_df['Variable'] != 'Constante'].copy().reset_index(drop=True)
y_pos   = np.arange(len(or_plot))
colors  = ['#E53935' if s in ['*','**','***'] else '#90A4AE' for s in or_plot['Sig.']]

for i, (_, row) in enumerate(or_plot.iterrows()):
    ax.errorbar(row['OR'], i,
                xerr=[[row['OR']-row['IC inf.']], [row['IC sup.']-row['OR']]],
                fmt='o', color=colors[i], ecolor=colors[i],
                capsize=5, markersize=9, linewidth=2)
    ax.text(or_plot['IC sup.'].max()*1.05, i,
            f"OR={row['OR']:.3f} {row['Sig.']}",
            va='center', fontsize=9)

ax.axvline(x=1, color='black', linestyle='--', linewidth=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(or_plot['Variable'], fontsize=10)
ax.set_xlabel("Odds Ratio (IC 95%)", fontsize=11)
ax.set_title("Forest Plot — Facteurs associés au désir d'avoir un autre enfant\n"
             "Régression logistique binaire (EDSC Cameroun 2018)",
             fontsize=12, fontweight='bold')
rouge = mpatches.Patch(color='#E53935', label='Significatif (p<0,05)')
gris  = mpatches.Patch(color='#90A4AE', label='Non significatif')
ref   = plt.Line2D([0],[0], color='black', linestyle='--', label='Référence OR=1')
ax.legend(handles=[rouge, gris, ref], loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig("../outputs/figures/figC_forest_plot_ameliore.png", bbox_inches='tight')
plt.show()
print("✓ Figure C sauvegardée")

✓ Figure C sauvegardée


In [146]:
# ── Figure D : Matrice de confusion ─────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm_labels = np.array([
    [f'VN = {tn:,}\n(Spécificité)', f'FP = {fp:,}'],
    [f'FN = {fn:,}',                 f'VP = {tp:,}\n(Sensibilité)']
])
im = ax.imshow([[tn,fp],[fn,tp]], interpolation='nearest', cmap='Blues')
ax.set_xticks([0,1])
ax.set_yticks([0,1])
ax.set_xticklabels(['Prédit : Ne désire pas','Prédit : Désire'], fontsize=9)
ax.set_yticklabels(['Réel : Ne désire pas','Réel : Désire'], fontsize=9)
for i in range(2):
    for j in range(2):
        val = [[tn,fp],[fn,tp]][i][j]
        ax.text(j, i, cm_labels[i,j], ha='center', va='center', fontsize=10,
                color='white' if val > max(tn,fp,fn,tp)/2 else 'black')
ax.set_title(f"Matrice de confusion — Régression logistique\n"
             f"% bien classés : {pct_ok:.1f}%  |  Sensib. : {sensib:.1f}%  |  Spécif. : {specif:.1f}%",
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/figD_matrice_confusion.png", bbox_inches='tight')
plt.show()
print("✓ Figure D sauvegardée")

✓ Figure D sauvegardée


In [147]:
# ── Table 5 : VIF ───────────────────────────────────────
X_vif = df_mod[VARS_MODELE].astype(float)

print(f"{'Variable':<28} {'VIF':>8}   Statut")
print("-" * 58)

vif_rows = []
for i, var in enumerate(VARS_MODELE):
    vif_val = variance_inflation_factor(X_vif.values, i)
    statut  = ("✓ OK (<5)" if vif_val < 5
               else ("⚠ Modéré (5-10)" if vif_val < 10
               else "✗ Problème (>10)"))
    label   = LABELS.get(var, var)
    print(f"{label:<28} {vif_val:>8.3f}   {statut}")
    vif_rows.append({'Variable':label, 'VIF':round(vif_val,3), 'Statut':statut})

vif_df = pd.DataFrame(vif_rows)
vif_df.to_csv("../outputs/tables/05_vif.csv", index=False)
print("\n✓ Sauvegardé → 05_vif.csv")

Variable                          VIF   Statut
----------------------------------------------------------
Âge                            18.583   ✗ Problème (>10)
Niveau d'instruction            7.109   ⚠ Modéré (5-10)
Nb enfants vivants              4.153   ✓ OK (<5)
Utilisation contraceptif        1.352   ✓ OK (<5)
Statut matrimonial              2.110   ✓ OK (<5)
Milieu de résidence             6.122   ⚠ Modéré (5-10)
Quintile de richesse           10.442   ✗ Problème (>10)
Emploi (travail)                3.074   ✓ OK (<5)
Région septentrionale           1.301   ✓ OK (<5)
Religion musulmane              1.542   ✓ OK (<5)

✓ Sauvegardé → 05_vif.csv


In [148]:
# ── Export multi-onglets ─────────────────────────────────
excel_path = "../outputs/tables/RESULTATS_COMPLETS_EDSC2018.xlsx"

# Reconstruire all_freq si nécessaire
all_freq = []
for nom, t in sections.items():
    first_col = t.columns[0]
    for _, row in t.iterrows():
        all_freq.append({'Variable':nom, 'Modalité':str(row[first_col]),
                         'Effectif':row['Effectif'], 'Pourcentage (%)':row['Pourcentage (%)']})

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

    # Résumé général
    pd.DataFrame([
        {'Information':'Effectif total',      'Valeur':f"{N:,}"},
        {'Information':'Désire un enfant',    'Valeur':f"{N_OUI:,} ({N_OUI/N*100:.1f}%)"},
        {'Information':'Ne désire pas',       'Valeur':f"{N_NON:,} ({N_NON/N*100:.1f}%)"},
        {'Information':'Source',              'Valeur':'EDSC Cameroun 2018'},
        {'Information':'Population cible',    'Valeur':'Femmes 15-49 ans'},
        {'Information':'Variables du modèle', 'Valeur':str(len(VARS_MODELE))},
    ]).to_excel(writer, sheet_name='Resume', index=False)

    # Stats continues
    desc.to_excel(writer, sheet_name='T1a-Stats continues')

    # Fréquences catégorielles
    pd.DataFrame(all_freq).to_excel(writer, sheet_name='T1b-Freq categories', index=False)

    # Tests Chi²
    chi2_df.to_excel(writer, sheet_name='T3a-Tests Chi2', index=False)

    # Mann-Whitney
    mw_df.to_excel(writer, sheet_name='T3b-Mann-Whitney', index=False)

    # Odds Ratios
    or_df.to_excel(writer, sheet_name='T4-Regression logistique', index=False)

    # Indicateurs du modèle
    pd.DataFrame(indic_rows).to_excel(writer, sheet_name='T4-Indicateurs modele', index=False)

    # VIF
    vif_df.to_excel(writer, sheet_name='T5-VIF', index=False)

print(f"✓ Fichier Excel créé : RESULTATS_COMPLETS_EDSC2018.xlsx")
print(f"  Onglets : Resume · T1a · T1b · T3a · T3b · T4-RL · T4-Indicateurs · T5-VIF")

✓ Fichier Excel créé : RESULTATS_COMPLETS_EDSC2018.xlsx
  Onglets : Resume · T1a · T1b · T3a · T3b · T4-RL · T4-Indicateurs · T5-VIF


In [149]:
# ── Résumé final dans le terminal ───────────────────────
print("=" * 55)
print("ANALYSE STATISTIQUE TERMINÉE")
print("=" * 55)
print(f"\nÉchantillon    : {N:,} femmes (15-49 ans)")
print(f"Désire         : {N_OUI:,} ({N_OUI/N*100:.1f}%)")
print(f"Ne désire pas  : {N_NON:,} ({N_NON/N*100:.1f}%)")
print(f"\nModèle logistique (10 variables) :")
print(f"  R² Nagelkerke : {nagelkerke:.4f}")
print(f"  AIC           : {logit_model.aic:.2f}")
print(f"  % bien classés: {pct_ok:.1f}%")
print(f"  Sensibilité   : {sensib:.1f}%")
print(f"  Spécificité   : {specif:.1f}%")
print(f"\nFichiers générés :")
for f in sorted(os.listdir("../outputs/tables/")):
    print(f"  📊 {f}")
for f in sorted(os.listdir("../outputs/figures/")):
    print(f"  🖼 {f}")

ANALYSE STATISTIQUE TERMINÉE

Échantillon    : 13,527 femmes (15-49 ans)
Désire         : 12,782 (94.5%)
Ne désire pas  : 745 (5.5%)

Modèle logistique (10 variables) :
  R² Nagelkerke : 0.2278
  AIC           : 4675.26
  % bien classés: 94.6%
  Sensibilité   : 99.9%
  Spécificité   : 3.1%

Fichiers générés :
  📊 01_stats_descriptives.csv
  📊 01a_stats_continues.csv
  📊 01b_stats_categories.csv
  📊 02_tests_chi2.csv
  📊 03_odds_ratio.csv
  📊 03a_tests_chi2.csv
  📊 03b_mann_whitney.csv
  📊 04_odds_ratio.csv
  📊 04_vif.csv
  📊 04b_indicateurs_modele.csv
  📊 05_comparaison_modeles.csv
  📊 05_vif.csv
  📊 06_comparaison_modeles_ML.csv
  📊 07_importance_variables.csv
  📊 RESULTATS_COMPLETS_EDSC2018.xlsx
  🖼 fig10_comparaison_modeles.png
  🖼 fig1_variable_cible.png
  🖼 fig2_desir_par_age.png
  🖼 fig3_desir_par_instruction.png
  🖼 fig4_desir_par_nb_enfants.png
  🖼 fig5_residence_contraceptif.png
  🖼 fig6_heatmap_correlation.png
  🖼 fig7_forest_plot_OR.png
  🖼 fig8_roc_curves.png
  🖼 fig9_impor